---

# Modelling Cascades in Cyber Systems

---

## Notebook 2.0: Networks, Their Structure and Metrics

## 1. Introduction

In Notebook 1, we lived entirely inside the (SOC) forest: a **Euclidean grid** (with **local neighbours** and **spatial correlations**). Real cyber systems look almost nothing like this.

This notebook is a conceptual pivot. Here, we shift from *spatial cascades* to *network cascades*. The underlying question:

> **If the internet is not a 2D grid, what *is* it — and how does that shape cascades?**

To answer that, we need one more foundational idea: graphs.

### 1.1 What graphs are, and why we use them

A **graph** is one of the simplest—and most powerful—abstractions in science and engineering. It consists of:

**Nodes (vertices)**: the things in the system
e.g. computers, servers, routers, user accounts, ASes, microservices…

**Edges (links)**: the relationships or interactions between them
e.g. communication channels, dependencies, trust relations, API calls…

This abstraction is deliberately flexible. A graph doesn’t care what a node is; it only cares *who is connected to whom*. That turns out to be enough structure to capture an extraordinary range of real systems.

Graphs are useful because they let us:

- **Visualise complex systems** and see patterns we can’t detect from raw data.

- **Quantify structure** with well-defined metrics (degree, clustering, path lengths…).

- **Model failures** and understand how disturbances travel through a network.

- **Compare synthetic networks** (idealised) with empirical ones (real data).

This notebook is not a course in graph theory, but we will lean on a few core ideas:

- **Degree** — how many neighbours a node has.

- **Clustering** — the tendency of neighbours to form triangles.

- **Path length** — how many hops it takes to move between nodes.

- **Connected components** — which parts of the graph are reachable from each other.

These ideas are simple, but their consequences are not. Real-world robustness, fragility, and cascade behaviour depend on them.

This notebook series uses a common set of network metrics to describe and compare graph structure; these are explained in the Appendix at the end of this notebook. These metrics help us understand how different network topologies influence cascade behaviour.


### 1.2 Why this matters for cyber systems

Cyber ecosystems—large-scale networks, corporate infrastructures, the internet—are **not spatial grids**. They are:

- **irregular**

- **heterogeneous**

- **highly clustered**

- **often have long-range shortcuts**

These structural differences profoundly change how cascades unfold. A single failure in a grid may stay local; the same failure in a scale-free or small-world network may travel far and fast.

So in this notebook we will:

1. **Build intuition** for different graph families (random, geometric, small-world, scale-free).

2. **Measure their structural properties** with simple metrics.

3. **Run small cascade experiments** and see how topology controls fragility.

4. **Develop a vocabulary and intuition** that we will carry into later, more cyber-realistic models.

Ultimately, we are moving from "cascades in space" to "cascades on networks", because this is the domain where cyber systems truly live.

---

## 2.1 Why Networks Matter for Cyber Cascades

The forest analogy has served us well, but here we gently dismantle it.

Real digital systems do **not** live on a spatial grid. Devices are not locked into fixed geometric neighbours.  
Instead, everything that matters flows along **logical edges**:

- Trust relationships  
- Authentication paths  
- VPN tunnels  
- API dependencies  
- Routing paths  
- Identity / access control relationships  
- Software supply-chain links  

This changes almost everything about cascades.

### Spatial vs Network Connectivity

| Property | Forest (Lattice) | Cyber System (Network) |
|---------|------------------|-------------------------|
| Neighbourhood | Fixed, local | Arbitrary: 1 hop may cross continents |
| Distance | Euclidean | “Hops” or dependency depth |
| Spread of failure | Local; geometric | Structural; controlled by graph topology |
| Correlation length | Physical radius | Component diameter / path lengths |
| Growth | Slow, uniform | Heterogeneous, bursty, rewired constantly |

### Key mental shift #1: Inter-node proximity is logical, not physical

In the SOC forest, the neighbours of a node were only those that were "physically" adjacent. Cascades remained local unless a spanning cluster formed and the system approached the percolation threshold.

In a network, two machines (nodes) separated by metres might be *logically distant*, while **two machines across the world might be *one hop apart***.

### Key mental shift #2: Inter-node connectivity is no longer bounded

In the SOC forest enforced a strict upper limit: every node had **at most four neighbours**. This geometric constraint prevented any node from becoming *disproportionately influential*.  

Real networks do not have this symmetry. In cyber systems, degree is *heterogeneous*:

- A personal device may have a handful of connections

- A corporate identity provider may have thousands  

- A cloud API gateway may have tens of thousands  

- A software supply-chain dependency can reach millions  

This means:

- **Connectivity is not uniform**  

- **Influence is unevenly distributed**  

- **Some nodes act as vulnerability amplifiers**  

- **Some nodes hardly matter at all**  

### Why this matters for cascades

When degree varies widely, so does cascade potential:

- A low-degree node fails → usually inconsequential  

- A medium-degree node fails → may disrupt a team or a service cluster  

- A high-degree node fails → ripples across the entire system  

- A *hub* fails → can trigger an internet-scale incident  

In other words:

> **Degree heterogeneity — not geometry — becomes the main control on cascade size.**

This is the conceptual doorway to scale-free networks, small-world shortcuts, and the structural fragilities we explore next.

> **Cascades in cyber systems are shaped by structure, not coordinates.**

This notebook introduces synthetic networks so we can explore these structural ideas cleanly before returning to cyber-realistic systems in later notebooks.

---

## 2.2 Generating Synthetic Networks

To understand cascading behaviour, we'll create a small *model zoo* of graphs. Each network type introduces different structural features:

- Degree distributions 

- Presence or absence of hubs  

- Clustering  

- Typical path lengths  

These properties profoundly affect how cascades unfold.

For each model, in subsequent notebooks, we will:

1. Generate the network  

2. Visualise it  

3. Plot its degree distribution  

4. Compute key metrics (clustering, diameter, path lengths)  

5. Discuss qualitative cascade behaviour



But, for now, we will do the introductions, beginning with the simplest.

---


### 2.2.1 Random Geometric Graphs (Spatial Networks)

Random Geometric Graphs (RGGs) are the closest conceptual analogue to the SOC forest model, but without the rigid structure of a lattice. They provide a natural first step from grid-based spatial systems to more general network-based SOC explorations.

In an RGG, nodes are placed randomly in two-dimensional space. **An edge forms between two nodes if their Euclidean separation is less than a fixed radius $r$.** Connectivity is therefore governed entirely by spatial proximity.

In the interactive figure below, use the sliders to explore how the network changes as parameters vary. In particular, **slowly increase $r$** and watch how the system transitions from isolated clusters to a globally connected network. This geometric percolation transition is central to understanding how cascades can propagate in spatially embedded systems. **The plot may not render via GitHub. If not, you can download the .ipynb.**

The plot title reports several simple but informative structural metrics (see the Appendix for more details):

- **$N$** — the number of nodes in the graph  
- **$r$** — the connection radius that defines which nodes are linked  
- **$M$** — the total number of edges present  
- **$\langle k \rangle$** — the mean node degree, defined as $\langle k \rangle = 2|E|/n$  
- **$LCC$** — the size of the *largest connected component*, i.e. the number of nodes in the largest mutually reachable cluster  

As $r$ increases, these quantities change rapidly near the connectivity threshold, offering an intuitive preview of the structural transitions explored more formally in later notebooks.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import ipywidgets as widgets
from IPython.display import display, clear_output


class RGGWidget:
    def __init__(self, n=250, seed=0, r=0.1, show_edges=True, domain=(0.0, 1.0, 0.0, 1.0)):
        self.domain = domain

        # Widgets
        self.n_slider = widgets.IntSlider(
            value=int(n), min=50, max=800, step=25, description="N", continuous_update=False
        )
        self.seed_slider = widgets.IntSlider(
            value=int(seed), min=0, max=9999, step=1, description="Seed", continuous_update=False
        )
        self.r_slider = widgets.FloatSlider(
            value=float(r), min=0.0, max=0.35, step=0.005, description="r", readout_format=".3f"
        )
        self.edges_toggle = widgets.Checkbox(value=bool(show_edges), description="show edges")
        self.metrics_toggle = widgets.Checkbox(value=True, description="show quick metrics")

        # Output areas
        self.status_out = widgets.Output()
        self.fig_out = widgets.Output()

        # Internal state
        self.xy = None
        self.dist2 = None
        self._segments = None
        self._triu_idx = None

        # Create figure once
        self.fig, self.ax = plt.subplots(figsize=(6.2, 6.2))
        self.ax.set_aspect("equal", adjustable="box")
        self.ax.set_xlim(self.domain[0], self.domain[1])
        self.ax.set_ylim(self.domain[2], self.domain[3])
        self.ax.set_xlabel("")
        self.ax.set_ylabel("")

        self.scatter = self.ax.scatter([], [], s=18)
        self.lc = LineCollection([], linewidths=0.6, alpha=0.45)
        self.ax.add_collection(self.lc)
        self.title = self.ax.set_title("Random Geometric Graph (RGG)")

        # ✅ Prevent the notebook from auto-displaying this figure (avoids the “static duplicate”)
        plt.close(self.fig)

        # Wire callbacks
        self.n_slider.observe(self._on_rebuild, names="value")
        self.seed_slider.observe(self._on_rebuild, names="value")
        self.r_slider.observe(self._on_update, names="value")
        self.edges_toggle.observe(self._on_update, names="value")
        self.metrics_toggle.observe(self._on_update, names="value")

        # Initial draw
        self._rebuild_geometry()
        self._update_plot()
        self._render_figure()

    def _render_figure(self):
        with self.fig_out:
            clear_output(wait=True)
            display(self.fig)

    def _rebuild_geometry(self):
        n = int(self.n_slider.value)
        seed = int(self.seed_slider.value)
        rng = np.random.default_rng(seed)

        xmin, xmax, ymin, ymax = self.domain
        self.xy = np.column_stack([
            rng.uniform(xmin, xmax, size=n),
            rng.uniform(ymin, ymax, size=n),
        ])

        dx = self.xy[:, 0][:, None] - self.xy[:, 0][None, :]
        dy = self.xy[:, 1][:, None] - self.xy[:, 1][None, :]
        self.dist2 = dx * dx + dy * dy

        self._triu_idx = np.triu_indices(n, k=1)
        i, j = self._triu_idx
        self._segments = np.stack([self.xy[i], self.xy[j]], axis=1)

        self.scatter.set_offsets(self.xy)

    def _edge_mask_for_r(self, r):
        r2 = float(r) * float(r)
        i, j = self._triu_idx
        return self.dist2[i, j] < r2

    def _quick_metrics(self, mask):
        n = self.xy.shape[0]
        m = int(mask.sum())
        mean_deg = (2 * m) / n if n else 0.0

        parent = np.arange(n, dtype=int)
        size = np.ones(n, dtype=int)

        def find(a):
            while parent[a] != a:
                parent[a] = parent[parent[a]]
                a = parent[a]
            return a

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra == rb:
                return
            if size[ra] < size[rb]:
                ra, rb = rb, ra
            parent[rb] = ra
            size[ra] += size[rb]

        if m > 0:
            i, j = self._triu_idx
            ii = i[mask]
            jj = j[mask]
            for a, b in zip(ii, jj):
                union(int(a), int(b))

        roots = np.array([find(k) for k in range(n)], dtype=int)
        _, counts = np.unique(roots, return_counts=True)
        largest_cc = int(counts.max()) if counts.size else 1

        return m, mean_deg, largest_cc

    def _update_plot(self):
        r = float(self.r_slider.value)
        show_edges = bool(self.edges_toggle.value)
        mask = self._edge_mask_for_r(r)

        self.lc.set_segments(self._segments[mask] if show_edges else [])

        n = self.xy.shape[0]
        if self.metrics_toggle.value:
            m, mean_deg, lcc = self._quick_metrics(mask)
            self.title.set_text(f"N = {n}, r = {r:.3f} | M = {m}, ⟨k⟩ = {mean_deg:.2f}, LCC = {lcc}")
        else:
            self.title.set_text(f"N = {n}, r = {r:.3f}")

        self.fig.canvas.draw()

    def _on_rebuild(self, change):
        with self.status_out:
            clear_output(wait=True)
            print("Rebuilding points & distances…")
        self._rebuild_geometry()
        self._update_plot()
        self._render_figure()
        with self.status_out:
            clear_output(wait=True)

    def _on_update(self, change):
        self._update_plot()
        self._render_figure()

    def ui(self):
        controls = widgets.HBox([
            widgets.VBox([self.r_slider]),#, self.edges_toggle, self.metrics_toggle
            widgets.VBox([self.n_slider, self.seed_slider]),            
            #widgets.VBox([self.r_slider, self.edges_toggle, self.metrics_toggle]),
        ])
        return widgets.VBox([controls, self.status_out, self.fig_out])


rgg = RGGWidget(n=250, seed=0, r=0.08, show_edges=True)
display(rgg.ui())



A detailed description can be found on the Wikipedia page for **[Random Geometric Graphs](https://en.wikipedia.org/wiki/Random_geometric_graph)**:  

#### Why this matters

RGGs capture the structure of systems where **physical distance directly constrains connectivity**. This makes them relevant to a wide range of real-world networks, including **IoT deployments, wireless and BLE networks, mesh networks, and industrial control systems**, where communication links are limited by range, interference, or line-of-sight.

From a SOC perspective, RGGs allow us to study how spatial embedding and locality shape cascade dynamics without imposing an artificial grid.

#### Structural Properties

- **Degree distribution**: Density-dependent, typically narrow for fixed $r$

- **Clustering**: High, due to geometric overlap of neighbourhoods

- **Connectivity threshold**: A percolation transition occurs once $r$ exceeds a critical value

- **Spatial interpretation**: Clusters and gaps are directly visible in physical space

#### Hypotheses for Cascade Behaviour

The points below are **structural intuitions rather than conclusions**. They motivate later analysis and will be tested explicitly in subsequent notebooks.

- Cascades are expected to be **strongly localised** relative to non-spatial network types

- **Geometric constraints** should limit long-range propagation, forcing cascades to follow spatial pathways

- **Heterogeneous node density** may create regions that act as local amplifiers, while sparse areas function as effective firebreaks

Whether these intuitions hold — and under what conditions they break down — is an empirical question explored later in the project.

---

We now shift from spatial networks, where geometry governs connectivity, to **logical networks** in which connections represent functional or organisational relationships rather than physical proximity.

---

### 2.2.2 Erdős–Rényi Graphs (Random Graphs)

Erdős–Rényi (ER) graphs provide the canonical baseline for network analysis. In an ER graph, each pair of nodes is connected independently with probability $p$, producing a network with minimal structure beyond what arises from randomness alone.

Despite their simplicity, ER graphs play a crucial role in network science: they define what “unstructured” connectivity looks like, and therefore what must be explained by more complex models.

More information on ER graphs can be found on Wikipedia **[here](https://en.wikipedia.org/wiki/Erdős–Rényi_model)**.

#### Structural Properties

- **Degree distribution**: Approximately Poisson for large $N$

- **Clustering**: Low, comparable to what would be expected by chance

- **Path length**: Short once the graph becomes connected

- **Connectivity threshold**: A giant connected component emerges when  
  $$
  p \gtrsim \frac{1}{N}
  $$

Above this threshold, a finite fraction of nodes becomes mutually reachable, even though the graph remains highly homogeneous.

#### Hypotheses for Cascade Behaviour

The following expectations arise directly from the structural simplicity of ER graphs and serve as reference points for later analysis:

- Cascades are expected to remain **small and short-lived** except near the percolation threshold

- The absence of hubs limits **amplification mechanisms** for large-scale events

- Cascade statistics should exhibit **narrow distributions** away from criticality  

These properties make ER graphs particularly useful as a control case when assessing whether observed SOC-like behaviour depends on more structured connectivity.

#### Why this matters

ER graphs provide a neutral reference against which other network types can be compared. Any deviation from ER-like behaviour — such as heavy-tailed cascade sizes or long-range correlations — must arise from structural features absent here, such as spatial embedding, clustering, or degree heterogeneity.

As such, ER graphs serve as an essential **null model** for identifying what is genuinely *special* about more structured networks, including small-world and scale-free graphs.

---

### 2.2.3 Watts–Strogatz Graphs (Small-World Networks)

Watts–Strogatz (WS) graphs interpolate between highly ordered lattices and fully random graphs. They are constructed by starting from a regular ring lattice and randomly rewiring a fraction of edges with probability $\beta$, introducing long-range shortcuts while largely preserving local structure.

This simple mechanism produces networks that combine **high clustering** with **short average path lengths** — a combination commonly observed in real-world systems.

More information about WS graphs can be found **[here](https://en.wikipedia.org/wiki/Watts–Strogatz_model)**. Duncan Watts, one of the discoverers of small-world networks authored an accessible book, *Six Degrees: The Science of a Connected Age*.

#### Structural Properties

- **Degree distribution**: Narrow, centred around the initial lattice degree

- **Clustering**: High for small $\beta$, remaining much larger than in ER graphs

- **Path length**: Drops rapidly as $\beta$ increases, even for small rewiring probabilities

- **Structure**: Local neighbourhoods connected by sparse long-range shortcuts  

For intermediate values of $\beta$, WS graphs occupy the so-called *small-world regime*, where local order coexists with global efficiency.

#### Hypotheses for Cascade Behaviour

The mixed structure of WS graphs leads to several plausible expectations, which will be tested explicitly in later notebooks:

- Local clustering may support **contained, lattice-like cascades** at small scales 

- Long-range shortcuts provide mechanisms for **rapid, non-local propagation**

- Cascades may exhibit **multi-scale behaviour**, combining local saturation with occasional system-spanning events

- Sensitivity to perturbations may increase sharply once shortcuts connect distant regions

In this sense, WS graphs represent a structurally plausible environment for SOC-like dynamics to emerge without relying on extreme degree heterogeneity.

#### Why this matters

Small-world networks are often cited as structurally realistic models for social, biological, and technological systems. In a cyber context, they resemble networks with strong local organisation (e.g. subnets or trust domains) linked by a small number of long-range connections.

From a SOC perspective, WS graphs are particularly interesting because they sit *between* order and randomness. By tuning $\beta$, we can explore how the introduction of shortcuts alters cascade statistics, correlation lengths, and the likelihood of large-scale events — all while maintaining a fixed underlying local structure.

This makes Watts–Strogatz graphs a natural bridge between spatially constrained systems and fully random networks.

---

### 2.2.4 Barabási–Albert Graphs (Scale-Free Networks)

Many real-world networks — including cyber and information systems — exhibit **scale-free structure**, in which a small number of nodes accumulate a disproportionately large number of connections. This structure often arises through *growth* and *preferential attachment*:

> New nodes tend to connect to nodes that are already well connected. Relatively well-connected nodes are termed **hubs**.

The Barabási–Albert (BA) model captures this mechanism explicitly by constructing the network incrementally, adding nodes one at a time and attaching them preferentially to existing high-degree nodes.

More on Barabási–Albert graphs han be found at **[this Wikipedia page](https://en.wikipedia.org/wiki/Barabási–Albert_model)**.

#### Structural Properties

- **Degree distribution**:  
  $$
  P(k) \sim k^{-3}
  $$  
  producing a heavy-tailed distribution in which hubs emerge naturally

- **Clustering**: Moderate, typically decreasing with system size

- **Path length**: Very short, often described as an “ultra-small world”

- **Heterogeneity**: Strong variation in node importance and load

These properties distinguish scale-free networks sharply from ER and WS graphs, which remain relatively homogeneous.

#### Hypotheses for Cascade Behaviour

The extreme degree heterogeneity of BA networks leads to several natural expectations, which motivate later analysis:

- Hubs may act as **powerful amplification points**, enabling cascades to propagate rapidly and over long distances

- The network is expected to be **robust to random failure**, as most randomly removed nodes have low degree

- Conversely, targeted removal or failure of hubs may trigger **disproportionately large cascades**

- Cascade size distributions may exhibit **heavy tails**, reflecting the underlying degree distribution

Whether these intuitions translate into SOC-like dynamics — such as scale-free event sizes emerging *from* the dynamics rather than being imposed by topology — is an open question explored later.

#### Why this matters

From a cyber perspective, Barabási–Albert graphs are highly evocative. Identity providers, cloud gateways, DNS infrastructure, API gateways, CI/CD pipelines, and centralised management services all function as hubs in real systems.

As a result, BA networks provide a structurally plausible setting for studying vulnerability amplification, systemic risk, and large-scale failure. Their growth-driven construction also resonates with how many cyber systems evolve over time, making them a particularly compelling candidate for exploring the relationship between network structure and SOC-like behaviour.

---

### 2.2.5 Hybrid Spatial + Heterogeneous Networks

Realistic cyber systems are rarely well described by a single graph family. Instead, they tend to combine **spatially constrained local structure** with **highly connected logical hubs**, reflecting both physical deployment and functional centralisation.

A useful conceptual model is therefore a hybrid network composed of a spatial layer coupled to a heterogeneous, hub-dominated layer.

#### Structural Composition

Such systems can be thought of as consisting of:

- **Local spatial clusters**, well approximated by Random Geometric Graphs  
  (e.g. sensors, IoT devices, embedded controllers, field equipment)  

- **A small number of high-degree logical nodes**, resembling scale-free hubs, representing:  
  - cloud services  
  - identity providers  
  - gateways and aggregation points  
  - update and orchestration servers  

Connections within the spatial layer are constrained by proximity, while links to hubs are largely unconstrained by geometry.

#### Why this matters

This hybrid perspective reflects how many cyber systems are actually built: physically distributed components interact locally, but depend on a small number of globally reachable services for coordination, identity, updates, and control.

Rather than introducing a new graph model at this stage, this section serves to **situate the earlier graph families within a broader modelling trajectory**. Later work may explore explicit hybrid constructions that combine spatial embedding, growth, and heterogeneity within a single framework.

In this sense, hybrid networks form a conceptual bridge between idealised graph models and the structure of real-world cyber systems.

---

## Summary and Next Steps

In this section, we have surveyed a small *model zoo* of graph families, ranging from spatially embedded networks to purely logical and heterogeneous structures. Each captures a different aspect of connectivity found in real systems, and each provides a distinct substrate on which cascades may unfold.

The purpose of this tour was not to argue that any single model *is* self-organised critical, but to establish a set of controlled environments in which that question can be asked systematically.

We now turn to the central task of the notebook: **investigating whether SOC-like behaviour emerges on these networks**, and if so, under what structural conditions. To do this, we apply many of the same analytical tools used for the SOC forest model — including cascade size statistics, scaling behaviour, and correlation measures — while also extending them to accommodate the specific features of logical networks, such as non-local propagation and heterogeneous node roles.

By holding the analysis framework as consistent as possible across graph types, we aim to isolate which structural features are merely incidental, and which are genuinely associated with SOC-like dynamics in networked systems.

---

## Appendix

### Glossary of Network Metrics

Different network topologies with identical $N$ and average degree can behave very differently under SOC-like cascades. Tracking these metrics allows us to quantify:

- how *local* or *global* connections are,  
- how quickly cascades can spread,  
- whether cascades concentrate in clusters,  
- and how structural differences lead to different cascade-size exponents.

#### **N — Number of nodes**
The total number of vertices in the graph.

Represents the number of devices, services, or entities in the system.


#### **M — Number of edges**
The total number of connections between nodes.

Represents communication links, trust relationships, wiring, or dependencies.


#### **Average degree (`avg_degree`)**
The mean number of neighbours per node:

$ \langle k \rangle = \frac{1}{N} \sum_i k_i $

Controls the overall density of the network.  
Higher values generally imply easier propagation of cascades.


#### **Standard deviation of degree (`std_degree`)**
Measures how *heterogeneous* the node degrees are.

- Low $\sigma$ → most nodes have similar degree (e.g., Watts–Strogatz).  
- High $\sigma$ → a few nodes have very high degree (e.g., Barabási–Albert hubs).

Degree heterogeneity strongly affects cascade amplification.

> "High" means something quite different in BA vs WS. A BA network with $\sigma = 15$ is normal; a WS network with $\sigma = 2$ may already be extreme.


#### **Connected components**
A **component** is a set of nodes where each pair is connected by some path.

- `num_components` — how many components exist  
- `size_lcc` — size of the **largest connected component** (LCC)  
- `fraction_in_lcc` — proportion of nodes inside the LCC

If `fraction_in_lcc` ≈ 1, the graph is globally well-connected.


#### **Clustering coefficient (`clustering`)**
Measures the tendency of a node’s neighbours to also be connected.

- High in spatial or social networks  
- Low in random or tree-like graphs

High clustering can *localise* cascades or create dense “pockets” of vulnerability.


#### **Diameter (`diameter_lcc`)**
The longest shortest-path distance between any two nodes in the LCC.

Represents the maximum number of hops required to traverse the network’s core cluster.


#### **Average path length (`avg_path_length_lcc`)**
The mean shortest-path distance between nodes in the LCC.

Lower values indicate “small-world” behaviour: failures can propagate quickly.

---
